# Reto F · Tolerancia a fallos (HDFS + Spark)

**Objetivo:** parar un DataNode mientras corre un job de Spark y documentar
1. qué hace **HDFS** (re-replicación de bloques) y
2. qué hace **Spark** (recuperación por linaje / reintentos).



## 0 · Antes de empezar (terminal)
Estado inicial ya guardado en `evidence/f_fsck_inicial.txt` y `evidence/f_report_inicial.txt`.

Comprobar que los tiempos de detección de nodo muerto son cortos (el compose los fija en unos segundos):
```
docker exec namenode hdfs getconf -confKey dfs.namenode.heartbeat.recheck-interval
docker exec namenode hdfs getconf -confKey dfs.heartbeat.interval
```
(por defecto serían 300000 ms y 3 s: ~10 min hasta dar por muerto un nodo).

In [1]:
from pyspark.sql import SparkSession, functions as F
import time, datetime as dt

HDFS = "hdfs://namenode:9000"
CURATED = f"{HDFS}/curated/viajes"

spark = (SparkSession.builder.appName("reto_F_tolerancia")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.hadoop.dfs.replication", "2")
         .getOrCreate())

def ahora(): return dt.datetime.now().strftime("%H:%M:%S")
print("Spark", spark.version, "| UI: http://localhost:4040")

Spark 4.2.0 | UI: http://localhost:4040


## 1 · Linaje de Spark (antes del fallo)
El **linaje** es el plan de transformaciones que Spark recuerda para cada DataFrame/RDD. Si se pierde una partición, Spark la **recalcula** repitiendo solo esos pasos desde la fuente (aquí, los bloques Parquet en HDFS) en vez de guardar copias del resultado.

In [2]:
df = spark.read.parquet(CURATED)
por_destino = (df.filter(~F.col("destino").rlike("^(PT|FR)") & (F.col("destino") != "externo"))
                 .groupBy("destino").agg(F.sum("viajes").alias("viajes")))

print("=== Plan (explain) ===")
por_destino.explain()
print("\n=== Linaje del RDD (toDebugString) ===")
print(por_destino.rdd.toDebugString().decode())

=== Plan (explain) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[destino#0], functions=[sum(viajes#12)])
   +- Exchange hashpartitioning(destino#0, 64), ENSURE_REQUIREMENTS, [plan_id=15]
      +- HashAggregate(keys=[destino#0], functions=[partial_sum(viajes#12)])
         +- Project [destino#0, viajes#12]
            +- Filter ((isnotnull(destino#0) AND NOT RLIKE(destino#0, ^(PT|FR))) AND NOT (destino#0 = externo))
               +- FileScan parquet [destino#0,viajes#12,fecha#22] Batched: true, DataFilters: [isnotnull(destino#0), NOT RLIKE(destino#0, ^(PT|FR)), NOT (destino#0 = externo)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/viajes], PartitionFilters: [], PushedFilters: [IsNotNull(destino), Not(EqualTo(destino,externo))], ReadSchema: struct<destino:string,viajes:double>



=== Linaje del RDD (toDebugString) ===
(1) MapPartitionsRDD[10] at javaToPython at DirectMethodHandleAccessor.java:103 []
 |  MapPart

## 2 · Referencia: resultado correcto antes del fallo
Guardamos el total y el nº de filas del curado. Después del fallo tienen que salir **idénticos**: demuestra que no se perdió ni cambió ningún dato.

In [3]:
ref_total = df.agg(F.sum("viajes")).first()[0]
ref_filas = df.count()
ref_dest  = por_destino.count()
print(f"[{ahora()}] REFERENCIA  filas={ref_filas:,}  suma(viajes)={ref_total:,.0f}  destinos={ref_dest:,}")

[11:49:52] REFERENCIA  filas=78,340,906  suma(viajes)=883,809,803  destinos=2,563


## 3 · Job largo + fallo del DataNode
Esta celda repite una agregación sobre `/curated/viajes` (lee desde HDFS cada vez) y va imprimiendo la hora.

**Cómo hacerlo:**
1. Ejecuta la celda de abajo.
2. Cuando salga la **iteración 3**, ve a la **terminal** y para un DataNode:
   ```
   docker stop datanode2
   ```
   Apunta la hora exacta (`date +%H:%M:%S`).
3. **No pares la celda.** Mira si el job sigue avanzando y si alguna iteración se retrasa.
4. Mientras corre, abre `http://localhost:4040` (Jobs/Stages: tareas reintentadas) y `http://localhost:9870` (Datanodes).

In [4]:
N_ITER = 20            
resultados = []
for i in range(1, N_ITER + 1):
    t0 = time.time()
    spark.sparkContext.setJobDescription(f"reto_F iteracion {i}")
    d = spark.read.parquet(CURATED)            # DataFrame nuevo: vuelve a leer de HDFS
    tot = (d.groupBy("fecha", "destino").agg(F.sum("viajes").alias("v"))
             .agg(F.sum("v")).first()[0])
    dur = time.time() - t0
    resultados.append((i, ahora(), round(dur, 1), tot))
    print(f"[{ahora()}] iteración {i:2d}: {dur:5.1f} s | suma(viajes)={tot:,.0f} | "
          f"{'OK' if abs(tot - ref_total) < 1 else 'DIFERENTE!'}")

[13:10:16] iteración  1:   4.7 s | suma(viajes)=883,809,803 | OK
[13:10:19] iteración  2:   3.3 s | suma(viajes)=883,809,803 | OK
[13:10:22] iteración  3:   3.1 s | suma(viajes)=883,809,803 | OK
[13:11:10] iteración  4:  47.5 s | suma(viajes)=883,809,803 | OK
[13:11:20] iteración  5:  10.5 s | suma(viajes)=883,809,803 | OK
[13:11:30] iteración  6:   9.7 s | suma(viajes)=883,809,803 | OK
[13:11:37] iteración  7:   6.7 s | suma(viajes)=883,809,803 | OK
[13:11:43] iteración  8:   6.4 s | suma(viajes)=883,809,803 | OK
[13:11:46] iteración  9:   3.3 s | suma(viajes)=883,809,803 | OK
[13:11:49] iteración 10:   3.0 s | suma(viajes)=883,809,803 | OK
[13:11:52] iteración 11:   2.7 s | suma(viajes)=883,809,803 | OK
[13:11:55] iteración 12:   3.5 s | suma(viajes)=883,809,803 | OK
[13:12:01] iteración 13:   6.0 s | suma(viajes)=883,809,803 | OK
[13:12:06] iteración 14:   4.1 s | suma(viajes)=883,809,803 | OK
[13:12:10] iteración 15:   4.0 s | suma(viajes)=883,809,803 | OK
[13:12:12] iteración 16: 

## 4 · Comprobar que el resultado sigue siendo correcto
Aunque un DataNode esté caído, el job debe dar **exactamente** el mismo resultado.

In [5]:
d = spark.read.parquet(CURATED)
post_total = d.agg(F.sum("viajes")).first()[0]
post_filas = d.count()
print(f"[{ahora()}] ANTES   filas={ref_filas:,}  suma={ref_total:,.0f}")
print(f"[{ahora()}] DESPUÉS filas={post_filas:,}  suma={post_total:,.0f}")
print("Resultado idéntico:", ref_filas == post_filas and abs(ref_total - post_total) < 1)

import pandas as pd
pd.DataFrame(resultados, columns=["iteracion", "hora", "segundos", "suma_viajes"])

[13:25:54] ANTES   filas=78,340,906  suma=883,809,803
[13:25:54] DESPUÉS filas=78,340,906  suma=883,809,803
Resultado idéntico: True


,iteracion,hora,segundos,suma_viajes
0,1,13:10:16,4.7,8.838098e+08
1,2,13:10:19,3.3,8.838098e+08
2,3,13:10:22,3.1,8.838098e+08
3,4,13:11:10,47.5,8.838098e+08
4,5,13:11:20,10.5,8.838098e+08
5,6,13:11:30,9.7,8.838098e+08
6,7,13:11:37,6.7,8.838098e+08
7,8,13:11:43,6.4,8.838098e+08
8,9,13:11:46,3.3,8.838098e+08
9,10,13:11:49,3.0,8.838098e+08


## 5 · Qué observar en HDFS (terminal)
Con `datanode2` parado y `datanode3` ya levantado (`docker compose --profile reto up -d datanode3`), repite cada pocos segundos:
```
docker exec namenode hdfs dfsadmin -report | grep -E "Live datanodes|Dead datanodes|Under replicated"
docker exec namenode hdfs fsck /curated/viajes | grep -E "Under-replicated|Status"
```
Los bloques *under-replicated* deben bajar a 0 al copiarse al `datanode3`. Guarda al final:
```
docker exec namenode hdfs fsck /curated/viajes -files -blocks -locations > evidence/f_fsck_final.txt
docker exec namenode hdfs dfsadmin -report > evidence/f_report_final.txt
```
Para restaurar: `docker start datanode2` y, al terminar, `docker stop datanode3`.

## 6 · Línea de tiempo y discusión (rellenar con tus horas)

| Hora | Evento | HDFS | Spark |
|---|---|---|---|
| __:__ | Estado inicial | 54 bloques, 2 réplicas, 2 DataNodes vivos | – |
| __:__ | Arranca `datanode3` | 3 DataNodes vivos | – |
| __:__ | Job en marcha | – | iteraciones normales |
| __:__ | `docker stop datanode2` | el NameNode deja de recibir latidos | tareas que leían de ese nodo cambian a otra réplica |
| __:__ | Nodo declarado muerto | bloques *under-replicated* | el job sigue y da el mismo resultado |
| __:__ | Re-replicación | copia de bloques al `datanode3` | – |
| __:__ | Todo sano | 54 bloques con 2 réplicas, 0 under-replicated | – |

**Preguntas que debe responder la discusión**
- ¿Cuánto tardó HDFS en detectar el nodo muerto y por qué (latidos / `recheck-interval`)?
- ¿De dónde copió las réplicas y cuánto tardó en volver a 2?
- ¿Se retrasó el job de Spark? ¿Hubo tareas reintentadas en el Spark UI?
- ¿Qué habría pasado sin `datanode3`? ¿Y con replicación 1?
- Diferencia entre tolerancia de HDFS (réplicas de **datos**) y de Spark (recalcular **cómputo** con el linaje).